<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">Los Embeddings Contextuales: ELMo y BERT</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">Los embeddings contextuales rompen con la idea de un único vector fijo por palabra. En este notebook se introduce cómo modelos como ELMo y BERT generan representaciones que dependen del contexto donde aparece cada término.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Una misma palabra puede significar cosas distintas según la frase, y esa variación necesita una representación más flexible.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A entender qué vuelve contextuales a estos embeddings y por qué supusieron un salto importante en NLP.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">En lenguaje, el significado no vive solo en la palabra aislada, sino en la relación con el contexto que la rodea.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Un embedding contextual cambia según la secuencia concreta en la que aparece el token.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Definición del texto
"""Este bloque define un texto de entrada que se procesará mediante embeddings contextuales.
La variable "texto" contiene una frase que se utilizará para obtener vectores de tokens dependientes del contexto.
El contenido permitirá observar cómo modelos como ELMo y BERT generan representaciones dinámicas."""

texto = "El banco está cerca del río pero el banco también es una entidad financiera."

texto # Muestra el texto de entrada definido para verificar su contenido.


In [ ]:
# Bloque 2 — Tokenización con BERT
"""Este bloque aplica la tokenización mediante el tokenizador de BERT.
Este bloque importa "AutoTokenizer" de "transformers" para utilizar un tokenizador preentrenado de Hugging Face.

Aplicamos la clase "AutoTokenizer", acompañada del método ".from_pretrained()", a la variable "tokenizer_bert" para cargar un tokenizador preentrenado.
El argumento "bert-base-uncased" especifica el modelo preentrenado que se desea utilizar.
La variable "tokenizer_bert" almacena el objeto tokenizador preentrenado cargado.

Aplicamos el método ".tokenize()" al objeto tokenizador preentrenado para obtener la lista de tokens de subpalabra correspondientes al texto de entrada.
El argumento "texto" es el texto de entrada que se desea tokenizar.
La variable "tokens_bert" almacena los tokens de subpalabra generados por el tokenizador.
Este tokenizador está entrenado para inglés, por lo que no es específico para el texto en español del ejemplo."""

from transformers import AutoTokenizer

tokenizer_bert = AutoTokenizer.from_pretrained("bert-base-uncased")
tokens_bert = tokenizer_bert.tokenize(texto)

tokens_bert # Muestra la lista de tokens de subpalabra generados por el tokenizador BERT.


In [ ]:
# Bloque 3 — Conversión del texto de entrada a IDs para BERT
"""Este bloque convierte el texto de entrada en identificadores numéricos.
Aplicamos el método "encode()" al tokenizador para obtener la secuencia de IDs correspondiente.
El argumento "texto" es la cadena de texto que queremos convertir en IDs.
El parámetro "add_special_tokens" agrega los tokens especiales que BERT necesita.

La variable "ids_bert" almacena la secuencia numérica resultante.
La conversión permite que los modelos de lenguaje trabajen con representaciones numéricas de texto, facilitando el procesamiento 
y la comprensión por parte de los algoritmos de aprendizaje automático."""

ids_bert = tokenizer_bert.encode(texto, add_special_tokens=True)

ids_bert # Muestra la secuencia de IDs numéricos correspondiente al texto para BERT.


In [ ]:
# Bloque 4 — Obtención de embeddings contextuales con BERT
"""Este bloque obtiene embeddings contextuales utilizando el modelo BERT.
Este bloque importa "torch" para trabajar con tensores en PyTorch.
Este bloque importa "AutoModel" de "transformers" para cargar modelos preentrenados de BERT.

Aplicamos la clase "AutoModel", acompañada del método ".from_pretrained()", a la variable "modelo_bert" para cargar un modelo preentrenado.
El argumento "bert-base-uncased" especifica el modelo preentrenado que se desea utilizar.
La variable "modelo_bert" almacena el modelo preentrenado de BERT cargado.

Aplicamos la función "torch.tensor()" a la lista "[ids_bert]" para obtener un tensor que pueda ser procesado por el modelo BERT.
El argumento "[ids_bert]" representa la secuencia de IDs de los tokens que se desean convertir en un tensor para ser procesados por el modelo BERT.

Asignamos la llamada a la instancia del modelo a la variable "salida_bert" para obtener la salida del modelo BERT.
El argumento "ids_bert_tensor" representa el tensor que contiene la secuencia de IDs de los tokens que se desea procesar con el modelo BERT.
La variable "salida_bert" contiene la salida completa del modelo BERT, incluidos los embeddings contextuales y otras posibles salidas.

Aplicamos el atributo ".last_hidden_state" a la variable "salida_bert" para obtener los embeddings contextuales generados por BERT para cada token.
La variable "embeddings_bert" contiene los vectores contextuales generados por BERT para cada token.
Esta operación extrae los embeddings contextuales generados por BERT para cada token de la secuencia de entrada."""

import torch
from transformers import AutoModel

modelo_bert = AutoModel.from_pretrained("bert-base-uncased")

ids_bert_tensor = torch.tensor([ids_bert])

salida_bert = modelo_bert(ids_bert_tensor)

embeddings_bert = salida_bert.last_hidden_state

embeddings_bert # Muestra los embeddings contextuales generados por BERT para cada token.


In [ ]:
# Bloque 5 — Selección de embeddings para una palabra polisémica
"""Este bloque selecciona los embeddings de las dos apariciones de la palabra "banco" en la frase.
Utilizamos una comprensión de lista para identificar las posiciones de la palabra "banco" en la secuencia de tokens "tokens_bert".

La comprensión de lista recorre la secuencia de tokens "tokens_bert" junto con sus índices.
La expresión "i + 1" suma 1 a cada índice para compensar el token especial `[CLS]` al inicio de la secuencia de IDs.
La subexpresión "for i, tok in enumerate(tokens_bert)" recorre cada token junto con su índice en la secuencia.
La función "enumerate()" devuelve un iterador de pares de índice y valor para cada elemento de la secuencia.
La condición "if tok == 'banco'" filtra los tokens que coinciden exactamente con la palabra "banco".
La variable "posiciones" contiene una lista con la posición de cada aparición de "banco" en la secuencia de tokens.

Aplicamos el operador de indexación "[]" a la variable "embeddings_bert" para seleccionar los embeddings correspondientes a las posiciones de la palabra "banco" en la secuencia de tokens.
Los índices "0" y "posiciones" representan, respectivamente, la primera secuencia del lote y las posiciones de la palabra "banco" en la secuencia de tokens.
La variable "embeddings_bank" contiene un vector contextual por cada aparición de "banco".
Esta búsqueda solo funciona si cada aparición se tokeniza exactamente como "banco", como ocurre en este ejemplo."""

posiciones = [i + 1 for i, tok in enumerate(tokens_bert) if tok == "banco"]

embeddings_bank = embeddings_bert[0, posiciones]

embeddings_bank # Muestra los embeddings de las distintas apariciones de la palabra «banco». 


In [ ]:
# Bloque 6 — Cálculo de similitud entre los embeddings contextuales
"""Este bloque calcula la similitud entre las dos apariciones de la palabra 'banco'.
Este bloque importa "torch.nn.functional" para calcular la similitud coseno entre los embeddings contextuales.

Aplicamos la función "F.cosine_similarity()" a los índices 0 y 1 de la secuencia de embeddings contextuales.
Los argumentos "embeddings_bank[0]" y "embeddings_bank[1]" representan, respectivamente, los índices 0 y 1 de la secuencia de vectores.
El parámetro "dim=0" indica que la similitud coseno se calcula a lo largo de la dimensión de los vectores.
La variable "similitud_bank" almacena el valor resultante del producto punto normalizado.
El cálculo compara las representaciones de ambas posiciones, pero por sí solo no demuestra que correspondan a sentidos distintos."""

import torch.nn.functional as F

similitud_bank = F.cosine_similarity(embeddings_bank[0], embeddings_bank[1], dim=0)

similitud_bank # Muestra la similitud coseno entre las dos apariciones de la palabra 'banco'.


In [ ]:
# Bloque 7 — Obtención de embeddings con ELMo
"""Este bloque obtiene embeddings contextuales utilizando ELMo.
Este bloque importa "ElmoEmbedder" de "allennlp.commands.elmo" para cargar modelos preentrenados de ELMo.

Aplicamos la clase "ElmoEmbedder()" a la variable "elmo" para crear una instancia del modelo ELMo preentrenado.
La variable "elmo" contiene la instancia del modelo ELMo preentrenado.

Aplicamos el método ".split()" a la variable "texto" para obtener una lista de tokens que será utilizada por ELMo.
La variable "tokens_elmo" contiene la lista de tokens obtenida al aplicar el método ".split()" al texto.

Aplicamos el método ".embed_sentence()" al modelo para obtener los embeddings de la secuencia de tokens.
El argumento "tokens_elmo" representa la lista de tokens obtenida mediante el método ".split()" aplicado al texto.
La variable "embeddings_elmo" contiene los embeddings generados por ELMo para cada token del texto.
ELMo produce tres capas de embeddings: la primera no es contextual y las dos siguientes incorporan información contextual.
Este ejemplo requiere AllenNLP y los archivos preentrenados de ELMo; el modelo predeterminado está entrenado en inglés."""

from allennlp.commands.elmo import ElmoEmbedder

elmo = ElmoEmbedder()
tokens_elmo = texto.split()

embeddings_elmo = elmo.embed_sentence(tokens_elmo)

embeddings_elmo # Muestra los embeddings obtenidos con ELMo.

In [ ]:
# Bloque 8 — Selección de embeddings ELMo para la palabra polisémica
"""Este bloque selecciona los embeddings de ELMo correspondientes a la palabra "banco".
Utilizamos una comprensión de lista para identificar las posiciones de la palabra "banco" en la secuencia de tokens "tokens_elmo".

La comprensión de lista recorre la secuencia de tokens "tokens_elmo" junto con sus índices.
La expresión "i" incorpora los índices de los tokens que cumplen la condición de la comprensión de lista.
La subexpresión "for i, tok in enumerate(tokens_elmo)" recorre cada token junto con su índice en la secuencia.
La función "enumerate()" produce pares formados por el índice y el valor de cada elemento de la secuencia.
La condición "if tok == 'banco'" filtra los tokens que coinciden exactamente con la palabra "banco".
La variable "posiciones_elmo" contiene una lista con los índices de los tokens que coinciden con la palabra "banco".
       
Aplicamos el operador de indexación "[]" a la variable "embeddings_elmo" para seleccionar los embeddings correspondientes a las posiciones de la palabra "banco".
Los índices ":" y "posiciones_elmo" representan, respectivamente, todas las capas y las posiciones de la palabra "banco" en la secuencia de tokens.
La variable "embeddings_banco_elmo" contiene los vectores de cada capa para cada aparición.
Este análisis permite inspeccionar los vectores de ambas posiciones; por sí solo no demuestra que tengan sentidos distintos."""

posiciones_elmo = [i for i, tok in enumerate(tokens_elmo) if tok == "banco"]      

embeddings_banco_elmo = embeddings_elmo[:, posiciones_elmo]     

embeddings_banco_elmo # Muestra los embeddings de las distintas apariciones de la palabra «banco». 


In [ ]:
# Bloque 9 — Resultados finales
"""Este bloque organiza los resultados del proceso de generación y análisis de embeddings contextuales.
La variable "resultado" almacena en forma de diccionario los tokens, las dimensiones de los embeddings de BERT y ELMo, los vectores de ELMo y la similitud de BERT.
La estructura permite visualizar cómo los modelos contextuales generan representaciones dinámicas dependientes del contexto."""
   
resultado = {
    "tokens_bert": tokens_bert,
    "embeddings_bert_shape": embeddings_bert.shape, # El atributo "shape" devuelve las dimensiones del tensor de embeddings de BERT.
    "similitud_bank": similitud_bank.item(),
    "embeddings_elmo_shape": embeddings_elmo.shape, # El atributo "shape" devuelve las dimensiones del array de embeddings de ELMo.
    "embeddings_banco_elmo": embeddings_banco_elmo
}

resultado # Muestra el diccionario con los resultados finales del análisis de embeddings contextuales.
   

<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto por qué los embeddings contextuales representan mejor la variabilidad real del lenguaje.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">Una palabra no tiene por qué conservar el mismo vector cuando cambia el contexto semántico o sintáctico.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Comparar embeddings contextuales como si funcionaran igual que los embeddings estáticos clásicos.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en los Modelos Recurrentes para Texto para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en los Modelos Recurrentes para Texto y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">03_Modelos_recurrentes_para_texto/01_Los_modelos_recurrentes_para_texto.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">